# Data.org Financial Health Prediction Challenge!

## Empowering Financial Inclusion Across Africa

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from numpy import random
from tqdm import tqdm
from pathlib import Path

from fastai.tabular.all import *
from ipywidgets import interact

from fastai.imports import *
np.set_printoptions(linewidth=130)
import gc

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score

import xgboost as xgb
from xgboost import plot_importance

import lightgbm as lgb
import catboost as cat

In [ ]:
path = Path('/kaggle/input/datasets/rubanzasilva/financial-health')
path

In [ ]:
train_df = pd.read_csv(path/'Train.csv',index_col='ID')
test_df = pd.read_csv(path/'Test.csv',index_col='ID')
sub_df = pd.read_csv(path/'SampleSubmission.csv')
vd_df = pd.read_csv(path/'VariableDefinitions.csv')

In [ ]:
!ls

## EDA

In [ ]:
train_df.head()

In [ ]:
train_df.describe().T

In [ ]:
train_df.info()

In [ ]:
missing = pd.DataFrame({
    'count': train_df.isnull().sum(),
    'pct': (train_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

In [ ]:
missing = pd.DataFrame({
    'count': test_df.isnull().sum(),
    'pct': (test_df.isnull().sum() / len(train_df) * 100).round(2)
})
print(missing[missing['count'] > 0])

In [ ]:
test_df

In [ ]:
# Number of unique categories
train_df['Target'].nunique()

# Count per category
train_df['Target'].value_counts()

## Prepare data for Machine Learning

In [ ]:
#splits = RandomSplitter(valid_pct=0.2)(range_of(original_df))
#train_df = pd.concat([train_df, original_df], ignore_index=True) *

cont_names,cat_names = cont_cat_split(train_df, dep_var='Target')

In [ ]:
splits = RandomSplitter(valid_pct=0.2)(range_of(train_df))

In [ ]:
to = TabularPandas(train_df, procs=[Categorify, FillMissing,Normalize],
                   cat_names = cat_names,
                   cont_names = cont_names,
                   y_names='Target',
                   y_block=CategoryBlock(),
                   splits=splits)

In [ ]:
cont_names

In [ ]:
dls = to.dataloaders(bs=64)

In [ ]:
dls.show_batch()

In [ ]:
learn = tabular_learner(dls, metrics=F1ScoreMulti(average='macro'))

In [ ]:
learn.fit_one_cycle(5)

In [ ]:
test_to = to.new(test_df)

In [ ]:
test_processed = test_to.items.copy()
test_processed.shape, test_df.shape

In [ ]:
missing = pd.DataFrame({
    'count': test_processed.isnull().sum(),
    'pct': (test_processed.isnull().sum() / len(test_processed) * 100).round(2)
})
print(missing[missing['count'] > 0])

In [ ]:
print("Missing values (%):")
print((test_processed.isnull().sum() / len(test_processed) * 100).round(2))

In [ ]:
test_new = test_df.dropna()
test_new.shape

In [ ]:
test_new

In [ ]:
test_processed['owner_age'].fillna(test_processed['owner_age'].median(), inplace=True)

In [ ]:
test_dl = dls.test_dl(test_processed)
test_dl

In [ ]:
test_dl.xs

In [ ]:
X_train, y_train = to.train.xs, to.train.ys.values.ravel()
X_test, y_test = to.valid.xs, to.valid.ys.values.ravel()

In [ ]:
learn = tabular_learner(dls, metrics=F1ScoreMulti)

In [ ]:
learn.fit_one_cycle(10)

In [ ]:
learn.show_results()

In [ ]:
row, clas, probs = learn.predict(df.iloc[0])

## Modelling

In [ ]:
%%time
xgb_clf = xgb.XGBClassifier()
xgb_clf

In [ ]:
xgb_clf_fit = xgb_clf.fit(X_train, y_train)
xgb_clf_fit

In [ ]:
#xgb_preds = xgb_clf_fit.predict(test_dl.xs)
xgb_sc_preds = xgb_clf_fit.predict(X_test)

In [ ]:
xgb_sc = f1_score(y_test, xgb_sc_preds, average='macro')
xgb_sc

In [ ]:
xgb_preds = xgb_clf_fit.predict(test_dl.xs)

In [ ]:
xgb_preds.shape, test_df.shape

In [ ]:
ss_df

In [ ]:
xgb_names = to.vocab[xgb_preds]
submit = ss_df
submit.Target = xgb_names
submit.to_csv('submission.csv',index=False)
submit

In [ ]:
!ls

In [ ]:
!rm submission.csv
sub_df = ss_df
sub_df['Target'] = xgb_preds
sub_df.to_csv('submission.csv', index=False)
sub = pd.read_csv('submission.csv')
sub

# F1 Score for Multiclass Classification

## Your Dataset Characteristics

**Target Classes:**
- Low: 6280 samples (65.3%)
- Medium: 2868 samples (29.8%)
- High: 470 samples (4.9%)

**Note**: Your dataset is imbalanced, especially the "High" class (only 4.9%). This makes F1 score the right choice over accuracy!

---

## F1 Score Averaging Methods

For multiclass problems, there are different ways to calculate F1:

### 1. **Macro F1** (Recommended for Competition)
```python
from sklearn.metrics import f1_score

# Calculate F1 for each class, then average (treats all classes equally)
f1_macro = f1_score(y_true, y_pred, average='macro')
```
- **Use when**: All classes are equally important (most competitions use this)
- Treats "High" class equally to "Low" class despite size difference
- **Most likely what your competition uses**

### 2. **Weighted F1**
```python
f1_weighted = f1_score(y_true, y_pred, average='weighted')
```
- **Use when**: Want to weight by class frequency
- Gives more importance to "Low" class (65%) than "High" (5%)

### 3. **Micro F1**
```python
f1_micro = f1_score(y_true, y_pred, average='micro')
```
- **Use when**: You care about overall performance
- Equals accuracy for multiclass problems

---

## Complete Implementation Example

### Using CatBoost (Recommended)

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from catboost import CatBoostClassifier, Pool
import pandas as pd

# Prepare data
X_train = train_df.drop('Target', axis=1)
y_train = train_df['Target']

# Identify categorical features
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

# Label encode target (High=0, Low=1, Medium=2 alphabetically)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Create validation split with stratification (important for imbalanced data!)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded  # Maintains class distribution
)

# Handle class imbalance with class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}
print(f"Class weights: {class_weights_dict}")

# Create CatBoost model optimized for F1
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro',  # Use Macro F1 for evaluation
    class_weights=class_weights_dict,  # Handle imbalance
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)

# Create data pools
train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

# Train
model.fit(train_pool, eval_set=val_pool)

# Validate
val_pred = model.predict(val_pool).flatten()
val_pred_labels = le.inverse_transform(val_pred.astype(int))
val_true_labels = le.inverse_transform(y_val)

# Calculate F1 scores
f1_macro = f1_score(val_true_labels, val_pred_labels, average='macro')
f1_weighted = f1_score(val_true_labels, val_pred_labels, average='weighted')

print(f"\nValidation Macro F1: {f1_macro:.4f}")
print(f"Validation Weighted F1: {f1_weighted:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(val_true_labels, val_pred_labels))

# Predict on test
test_pool = Pool(test_df, cat_features=cat_features)
test_pred = model.predict(test_pool).flatten()
test_pred_labels = le.inverse_transform(test_pred.astype(int))

# Create submission
submission = pd.DataFrame({
    'ID': test_df.index,
    'Target': test_pred_labels
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
```

---

## Using XGBoost with F1 Optimization

```python
import xgboost as xgb
from sklearn.metrics import f1_score

# Custom F1 evaluation function for XGBoost
def f1_eval_macro(preds, dtrain):
    labels = dtrain.get_label()
    # Reshape predictions for multiclass
    preds_reshaped = preds.reshape(len(np.unique(labels)), -1).T
    preds_class = np.argmax(preds_reshaped, axis=1)
    f1 = f1_score(labels, preds_class, average='macro')
    return 'f1_macro', f1

# Prepare data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_encoded, test_size=0.2, random_state=42, stratify=y_train_encoded
)

# Handle categorical features
cat_cols = X_train.select_dtypes(include=['object']).columns
for col in cat_cols:
    le_col = LabelEncoder()
    X_tr[col] = le_col.fit_transform(X_tr[col].astype(str))
    X_val[col] = le_col.transform(X_val[col].astype(str))
    test_df[col] = le_col.transform(test_df[col].astype(str))

# Create DMatrix
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_val, label=y_val)

# Parameters
params = {
    'objective': 'multi:softmax',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'max_depth': 6,
    'eta': 0.05,
    'seed': 42
}

# Train with F1 monitoring
evals = [(dtrain, 'train'), (dval, 'val')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    feval=f1_eval_macro,
    early_stopping_rounds=50,
    verbose_eval=50
)

# Predict
dtest = xgb.DMatrix(test_df)
predictions = model.predict(dtest)
predicted_labels = le.inverse_transform(predictions.astype(int))
```

---

## Using LightGBM with F1 Optimization

```python
import lightgbm as lgb
from sklearn.metrics import f1_score

# Custom F1 metric for LightGBM
def lgb_f1_macro(y_true, y_pred):
    # y_pred is a 1D array of predicted probabilities for all classes
    y_pred = y_pred.reshape(len(np.unique(y_true)), -1).T
    y_pred_class = np.argmax(y_pred, axis=1)
    f1 = f1_score(y_true, y_pred_class, average='macro')
    return 'f1_macro', f1, True  # True means higher is better

# Prepare data (same encoding as XGBoost above)
# ... encode categorical features ...

# Create datasets
train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Parameters
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'max_depth': 6,
    'seed': 42,
    'verbose': -1
}

# Train
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    feval=lgb_f1_macro,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)
```

---

## Cross-Validation with F1 Score

For more robust evaluation:

```python
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# Prepare data
X = train_df.drop('Target', axis=1)
y = le.fit_transform(train_df['Target'])

cat_features = X.select_dtypes(include=['object']).columns.tolist()

# 5-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold + 1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    # Train model
    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function='MultiClass',
        eval_metric='TotalF1:average=Macro',
        cat_features=cat_features,
        random_seed=42,
        verbose=0
    )

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)

    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)

    # Predict and evaluate
    val_pred = model.predict(val_pool).flatten()
    f1 = f1_score(y_val, val_pred.astype(int), average='macro')
    f1_scores.append(f1)

    print(f"Fold {fold + 1} Macro F1: {f1:.4f}")

print(f"\nMean Macro F1: {np.mean(f1_scores):.4f} (+/- {np.std(f1_scores):.4f})")
```

---

## Tips for Improving F1 Score

### 1. Handle Class Imbalance
```python
# Option A: Class weights (already shown above)
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)

# Option B: SMOTE (Synthetic Minority Over-sampling)
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_encoded, y_train)

# Option C: Adjust decision threshold (for probability predictions)
# Get probabilities instead of hard predictions
probs = model.predict_proba(val_pool)
# Adjust thresholds for each class to maximize F1
```

### 2. Focus on Minority Class (High)
```python
# Feature engineering specifically for rare class
# Analyze what makes "High" class different
high_samples = train_df[train_df['Target'] == 'High']
print(high_samples.describe())
```

### 3. Ensemble Models
```python
# Combine predictions from multiple models
from sklearn.ensemble import VotingClassifier

# Use soft voting (averages probabilities)
ensemble = VotingClassifier(
    estimators=[
        ('cat', catboost_model),
        ('xgb', xgboost_model),
        ('lgb', lightgbm_model)
    ],
    voting='soft'
)
```

---

## Quick Reference: F1 Score by Library

| Library | Metric Parameter | Value |
|---------|-----------------|-------|
| CatBoost | `eval_metric` | `'TotalF1:average=Macro'` |
| XGBoost | Custom `feval` | Use f1_score function |
| LightGBM | Custom `feval` | Use f1_score function |
| scikit-learn | `f1_score()` | `average='macro'` |

---

## What to Check in Competition Rules

Look for phrases like:
- "macro-averaged F1"
- "mean F1"
- "F1 score averaged across all classes"

This will tell you which averaging method to use!
